# 00 · Organização do Ambiente
### Projeto Prático — Arquitetura Medalhão com a base CineData (TMDB/IMDb)

Objetivos deste notebook:
1. Entender a estrutura de diretórios/tabelas que vamos usar (Landing → Bronze → Silver → Gold)
2. Criar o `catalog` e os `schemas` no Unity Catalog
3. Criar o **Volume** que representa a nossa zona de *Landing*
4. Validar que os arquivos `.csv` de origem do acervo de filmes estão disponíveis

Este notebook é reaproveitado pelos demais (`%run ./00_Organizacao_do_Ambiente`), então evite alterar os nomes das variáveis abaixo sem ajustar os outros notebooks.


## 1. Parâmetros do ambiente, Criação do Catalog, Schemas e Volume

In [0]:
# Célula 2: Criação do Catalog, Schemas (com prefixo cinedata_) e Volume

# 1. Ajuste dos nomes dos schemas com o prefixo cinedata_
catalog = "workspace"  # Mantendo no catalog padrão do workspace 
bronze_schema_name = "cinedata_bronze"
silver_schema_name = "cinedata_silver"
gold_schema_name = "cinedata_gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# Caminho da landing zone no Volume
landing_path = "/Volumes/workspace/default/landing/cinedata" #caminho de pastas criado por mim 

# 2. Execução dos comandos SQL de criação (caso não existam)
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

# Criação do Volume para a Landing Zone dentro do schema Bronze
spark.sql(f"CREATE VOLUME IF NOT EXISTS {bronze_schema}.landing")

print("Catalog, schemas com prefixo 'cinedata_' e volume criados/verificados com sucesso!")
print(f"Bronze: {bronze_schema}")
print(f"Silver: {silver_schema}")
print(f"Gold:   {gold_schema}")

Catalog, schemas com prefixo 'cinedata_' e volume criados/verificados com sucesso!
Bronze: workspace.cinedata_bronze
Silver: workspace.cinedata_silver
Gold:   workspace.cinedata_gold


%md
## 3. Upload dos arquivos de origem

Vamos importar os **5 arquivos** que serão utilizados no Databricks (projeto CineData) em **Catalog** → `workspace` → `cinedata_bronze` → `landing` → **Upload to this volume**[cite: 5, 6, 7]:
   - `credits_and_tags_IMDB_TMDB.csv`
   - `movies_financials_IMDB_TMDB.csv`
   - `movies_info_TMDB_IMDB.csv`
   - `movies_metrics_IMDB_TMDB.csv`
   - `movies_reviews.csv`

## 4. Validação da Landing Zone

In [0]:
expected_files = [
    "credits_and_tags_IMDB_TMDB.csv",
    "movies_financials_IMDB_TMDB.csv",
    "movies_info_TMDB_IMDB.csv",
    "movies_metrics_IMDB_TMDB.csv",
    "movies_reviews.csv"
]

try:
    existing = {f.name for f in dbutils.fs.ls(landing_path)}
except Exception as e:
    existing = set()
    print(f"[ALERTA] Não foi possível listar '{landing_path}'. Faça o upload dos arquivos antes de continuar.\n{e}")

missing = [f for f in expected_files if f not in existing]

if missing:
    print("[PENDENTE] Arquivos ainda não encontrados na landing zone:")
    for f in missing:
        print(f"  - {f}")
else:
    print("[OK] Todos os arquivos esperados estão na landing zone:")
    for f in dbutils.fs.ls(landing_path):
        print(f"  - {f.name} ({f.size/1024:.1f} KB)")

[OK] Todos os arquivos esperados estão na landing zone:
  - cotacao_dolar.json (95.5 KB)
  - credits_and_tags_IMDB_TMDB.csv (21797.5 KB)
  - movies_financials_IMDB_TMDB.csv (1433.3 KB)
  - movies_info_TMDB_IMDB.csv (33129.0 KB)
  - movies_metrics_IMDB_TMDB.csv (3246.0 KB)
  - movies_reviews.csv (1878.4 KB)
